# 02 — Normalize and QC: Minimal-QC Engineering Scaffold

Claude P2 precontract v1.1(`b9990af`)을 소비한 notebook-first scaffold다. 전체 565만 행 실행은 승인되지 않았으며, 이 notebook은 synthetic preflight만 수행한다.

## 0 Contract / Scope


In [ ]:
SYNTHETIC_ONLY = True
FULL_RUN_AUTHORIZED = False
PHASE = '02'
UNRESOLVED_RESEARCH_ITEMS = {'manual_audit_logistics', 'cr003_documentation'}
assert SYNTHETIC_ONLY and not FULL_RUN_AUTHORIZED


## 1 Environment + Inputs


In [ ]:
import json
from tokenization_premium.paths import PROJECT_ROOT
from tokenization_premium.phase2 import (
    BLOCKED_BY_P2_CONTRACT, P2_CONTRACT_COMMIT, PAIR_REGISTRY_V001_RELATIVE_PATH,
    PAIR_REGISTRY_V002_RELATIVE_PATH, decode_integrity_ok, derive_pair_quality_status, empty_text_flag,
    iter_parquet_batches, language_side_anomaly_review, manual_audit_import_schema, named_entity_deferred_fields,
    normalize_ssot_text, open_phase2_duckdb, select_analysis_representative_pair_id,
    validate_d01_manifest_handoff, write_parquet_batches_atomic,
)
from tokenization_premium.progress import ProgressHeartbeat, progress_tqdm

assert P2_CONTRACT_COMMIT == 'b9990afbf3fc0ed2a5e80fb4def1565e9ba3ebf4'
PAIR_REGISTRY_V001 = PROJECT_ROOT / PAIR_REGISTRY_V001_RELATIVE_PATH
PAIR_REGISTRY_V002 = PROJECT_ROOT / PAIR_REGISTRY_V002_RELATIVE_PATH
RUNTIME_DIR = PROJECT_ROOT / '.runtime/p2-duckdb-synthetic'
connection = open_phase2_duckdb(RUNTIME_DIR)
try:
    assert connection.execute('SELECT 1').fetchone() == (1,)
finally:
    connection.close()
RUNTIME_PREFLIGHT = 'SYNTHETIC_PASS'
RUNTIME_PREFLIGHT


## 2 D-01 Integrity Handoff


In [ ]:
D01_MANIFEST_PATH = PROJECT_ROOT / 'outputs/manifests/PAIR_REGISTRY_MANIFEST_v001.json'
D01_ORACLE_PATH = PROJECT_ROOT / 'outputs/manifests/data_recon/G1_INGEST_EXPECTATIONS_v001.json'
d01_manifest = json.loads(D01_MANIFEST_PATH.read_text(encoding='utf-8'))
d01_oracle = json.loads(D01_ORACLE_PATH.read_text(encoding='utf-8'))
d01_validation = validate_d01_manifest_handoff(d01_manifest, d01_oracle)
D01_ROW_COUNT = d01_validation.row_count
D01_HANDOFF = {
    'input': str(PAIR_REGISTRY_V001_RELATIVE_PATH),
    'prospective_output': str(PAIR_REGISTRY_V002_RELATIVE_PATH),
    'manifest_validation': d01_validation.status,
    'manifest_rows': D01_ROW_COUNT,
    'manifest_sha256': d01_manifest['pair_registry']['sha256'],
    'pair_id_distinct': d01_validation.pair_id_distinct,
    'canonical_input_count': d01_validation.canonical_input_count,
    'full_scan': 'NOT_RUN_REVISION',
}
PROSPECTIVE_LONG_STAGES = [
    {'phase': PHASE, 'stage': 'NORMALIZATION', 'total': D01_ROW_COUNT, 'checkpoint': 'NORMALIZATION_BATCH'},
    {'phase': PHASE, 'stage': 'QC_FLAG_COMPUTATION', 'total': D01_ROW_COUNT, 'checkpoint': 'QC_FLAG_BATCH'},
    {'phase': PHASE, 'stage': 'DUPLICATE_DISPOSITION', 'total': None, 'checkpoint': 'DUPLICATE_GROUP_BATCH'},
    {'phase': PHASE, 'stage': 'LANG_SIDE_SMOKE', 'total': D01_ROW_COUNT, 'checkpoint': 'LANG_SIDE_BATCH'},
    {'phase': PHASE, 'stage': 'DECODE_INTEGRITY', 'total': D01_ROW_COUNT, 'checkpoint': 'DECODE_BATCH'},
    {'phase': PHASE, 'stage': 'MANUAL_AUDIT_IMPORT', 'total': 500, 'checkpoint': 'MANUAL_AUDIT_ROW'},
    {'phase': PHASE, 'stage': 'QC_FLOW', 'total': None, 'checkpoint': 'QC_FLOW_STAGE'},
    {'phase': PHASE, 'stage': 'REGISTRY_V002_WRITE', 'total': None, 'checkpoint': 'REGISTRY_V002_BATCH'},
    {'phase': PHASE, 'stage': 'ARTIFACT_HASH', 'total': None, 'checkpoint': 'ARTIFACT_HASH_CHUNK'},
]
D01_HANDOFF, PROSPECTIVE_LONG_STAGES


## 3 Normalization

SSOT-frozen operations only: NFC, edge-only BOM removal, outer trim. Internal whitespace and internal U+FEFF/zero-width evidence are preserved.


In [ ]:
synthetic_inputs = ['e\u0301', '\ufeff문장\ufeff', '텍스\ufeff트', '내부  공백', 'MiXeD ＡＢＣ']
normalization_results = []
with ProgressHeartbeat(
    run_id='p2-scaffold-synthetic', phase=PHASE, stage='NORMALIZATION_SYNTHETIC',
    total=len(synthetic_inputs),
) as heartbeat:
    for value in progress_tqdm(synthetic_inputs, desc='P2 synthetic normalization', position=1, leave=False):
        normalization_results.append(normalize_ssot_text(value))
        heartbeat.update(1)
    heartbeat.checkpoint('NORMALIZATION_SYNTHETIC_COMPLETE', normalized_rows=len(normalization_results))
normalization_results


## 4 QC Flag Computation


In [ ]:
synthetic_structural_flags = {
    'empty_text_flag': empty_text_flag('한국어', 'English'),
    'decode_integrity_flag': not decode_integrity_ok('readable Unicode'),
    'markup_dominant_flag': False, 'control_char_excess_flag': False,
    'exact_duplicate_flag': False,
}
SYNTHETIC_PAIR_STATUS = derive_pair_quality_status(synthetic_structural_flags)
synthetic_structural_flags, SYNTHETIC_PAIR_STATUS


## 5 Duplicate Disposition


In [ ]:
synthetic_duplicate_group = [
    {'pair_id': 'pair_000_legacy', 'primary_analysis_eligible': False},
    {'pair_id': 'pair_900_tier_a', 'primary_analysis_eligible': True},
]
analysis_representative_pair_id = select_analysis_representative_pair_id(synthetic_duplicate_group)
assert analysis_representative_pair_id == 'pair_900_tier_a'
analysis_representative_pair_id


## 6 Language-Side Smoke

Model LID나 confidence score가 아니다. Unicode count 기반 review-only wiring sanity check이며 자동 배제하지 않는다.


In [ ]:
language_smoke_examples = {
    'ko_mixed': language_side_anomaly_review('OpenAI API 사용 방법', expected_side='KO'),
    'en_mixed': language_side_anomaly_review('Korea 서울', expected_side='EN'),
    'ko_clear_opposite': language_side_anomaly_review('This is clearly English text', expected_side='KO'),
}
assert derive_pair_quality_status({'lang_side_anomaly_review_flag': True}) == 'accepted'
language_smoke_examples


## 7 Semantic QC


In [ ]:
MANUAL_AUDIT_SCHEMA = manual_audit_import_schema()
MANUAL_AUDIT_LOGISTICS = BLOCKED_BY_P2_CONTRACT
POPULATION_EMBEDDING_SIMILARITY = 'NOT_IMPLEMENTED'
assert {'manual_semantic_score', 'manual_language_side_status', 'manual_audit_status'} <= set(MANUAL_AUDIT_SCHEMA.names)
MANUAL_AUDIT_SCHEMA, MANUAL_AUDIT_LOGISTICS, POPULATION_EMBEDDING_SIMILARITY


## 8 QC Flow


In [ ]:
QC_FLOW_POLICY = {
    'hard_gate': ['empty_text_flag', 'decode_integrity_flag', 'markup_dominant_flag',
                  'control_char_excess_flag', 'exact_duplicate_flag'],
    'review_only': ['lang_side_anomaly_review_flag'],
    'automatic_lid_rejection': False,
}
QC_FLOW_POLICY


## 9 Registry v002


In [ ]:
REGISTRY_V002_STATUS = 'SCAFFOLD_ONLY_NO_ARTIFACT'
NAMED_ENTITY_FIELDS = named_entity_deferred_fields()
STREAMING_INFRA = {
    'iterator': iter_parquet_batches.__name__,
    'atomic_writer': write_parquet_batches_atomic.__name__,
    'full_v002_generated': False,
}
REGISTRY_V002_STATUS, STREAMING_INFRA, NAMED_ENTITY_FIELDS


## 10 Artifact / Hash


In [ ]:
ARTIFACT_HASH_STATUS = BLOCKED_BY_P2_CONTRACT
ATOMICITY_PATTERN = '*.partial -> validate -> os.replace'
FINAL_QC_ARTIFACT_GENERATED = False
ARTIFACT_HASH_STATUS, ATOMICITY_PATTERN, FINAL_QC_ARTIFACT_GENERATED


## 11 G1 Closure Evidence


In [ ]:
G1_CLOSURE_EVIDENCE_STATUS = 'NOT_GENERATED_NO_FULL_RUN'
SCAFFOLD_VERDICT = {
    'engineering_scaffold': 'READY',
    'research_contract': f'CONSUMED_{P2_CONTRACT_COMMIT}',
    'manual_audit_logistics': BLOCKED_BY_P2_CONTRACT,
    'g1_gate': 'OPEN',
}
G1_CLOSURE_EVIDENCE_STATUS, SCAFFOLD_VERDICT
